In [1]:

import os
import glob
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torchvision.transforms.v2 as T
 
BATCH_SIZE = 8
EPOCHS = 20
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device :", device)

Device : mps


In [4]:
inv = pd.read_csv("inventaire_oasis2.csv")
tissue_3d = pd.read_csv("biomarkers_tissue_volumes.csv")
biomarkers_2d = pd.read_csv("biomarkers_2D.csv")
 
clinical_path = glob.glob("*longitudinal*.xlsx") + glob.glob(
    os.path.join("..", "src", "Data", "*longitudinal*.xlsx"))
clinical = pd.read_excel(clinical_path[0])
 
first_visit = clinical[clinical["Visit"] == 1].copy()
first_visit = first_visit[first_visit["Group"].isin(["Nondemented", "Converted"])]
first_visit = first_visit.rename(columns={"MRI ID": "subject_id"})
first_visit["label"] = first_visit["Group"].map({"Nondemented": 0, "Converted": 1})

# fusion image (inv) + biomarqueurs 3D + biomarqueurs 2D + label
merged = first_visit[["subject_id", "label"]].merge(inv, on="subject_id", how="inner")
merged = merged[merged["has_any_volume"]]
merged = merged.merge(tissue_3d, on="subject_id", how="inner")
merged = merged.merge(biomarkers_2d, on="subject_id", how="inner")
merged = merged.reset_index(drop=True)
print(f"{len(merged)} sujets avec image + biomarqueurs 3D + 2D disponibles.")
print(merged["label"].value_counts())
 
BIOMARKER_FEATURES = [
    "gray_matter_mL", "white_matter_mL", "csf_mL", "total_brain_mL",
    "asymmetry_gray_matter", "asymmetry_white_matter",
    "intensity_mean", "intensity_std", "texture_contrast",
]

86 sujets avec image + biomarqueurs 3D + 2D disponibles.
label
0    72
1    14
Name: count, dtype: int64


In [5]:
def load_central_slice(subject_path):
    img_files = sorted(glob.glob(os.path.join(subject_path, "**", "mpr-*.nifti.img"), recursive=True))
    volumes = [np.squeeze(nib.load(f).get_fdata()) for f in img_files]
    volume = np.mean(volumes, axis=0)
    return volume[:, :, volume.shape[2] // 2]
 
 
print("\nPréchargement des images en mémoire...")
image_cache = {}
for _, row in merged.iterrows():
    img = load_central_slice(row["path"])
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
    img = torch.nn.functional.interpolate(img.unsqueeze(0), size=(128, 128)).squeeze(0)
    image_cache[row["subject_id"]] = img
print(f"{len(image_cache)} images préchargées.")


Préchargement des images en mémoire...
86 images préchargées.


In [6]:
class HybridDataset(Dataset):
    def __init__(self, df, biomarker_array):
        self.df = df.reset_index(drop=True)
        self.biomarkers = biomarker_array  # déjà standardisé (StandardScaler)
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = image_cache[row["subject_id"]]
        biomarker_vec = torch.tensor(self.biomarkers[idx], dtype=torch.float32)
        return img, biomarker_vec, row["label"]

In [7]:
class HybridCNN(nn.Module):
    def __init__(self, n_biomarkers):
        super().__init__()
        self.cnn_branch = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),  # -> vecteur de 32 valeurs
        )
        self.biomarker_branch = nn.Sequential(
            nn.Linear(n_biomarkers, 16), nn.ReLU(), nn.Dropout(0.3),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(32 + 16, 2),  # fusion des deux branches
        )
 
    def forward(self, img, biomarkers):
        img_features = self.cnn_branch(img)
        biomarker_features = self.biomarker_branch(biomarkers)
        combined = torch.cat([img_features, biomarker_features], dim=1)
        return self.classifier(combined)
 
 
train_augment = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05)),
])

In [8]:
X_idx = merged.index.values
y = merged["label"].values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
 
all_preds, all_labels, all_probs = [], [], []
 
for fold, (train_idx, val_idx) in enumerate(skf.split(X_idx, y)):
    train_df = merged.iloc[train_idx].reset_index(drop=True)
    val_df = merged.iloc[val_idx].reset_index(drop=True)
 
    # standardisation des biomarqueurs : fit sur le train du fold uniquement
    # (jamais sur le val, pour éviter toute fuite d'information)
    scaler = StandardScaler()
    train_biomarkers = scaler.fit_transform(train_df[BIOMARKER_FEATURES].fillna(train_df[BIOMARKER_FEATURES].mean()))
    val_biomarkers = scaler.transform(val_df[BIOMARKER_FEATURES].fillna(train_df[BIOMARKER_FEATURES].mean()))
 
    class_counts = train_df["label"].value_counts()
    weights = torch.tensor([1.0 / class_counts.get(0, 1), 1.0 / class_counts.get(1, 1)], dtype=torch.float32)
 
    train_loader = DataLoader(HybridDataset(train_df, train_biomarkers), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(HybridDataset(val_df, val_biomarkers), batch_size=BATCH_SIZE)
 
    model = HybridCNN(n_biomarkers=len(BIOMARKER_FEATURES)).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
 
    for epoch in range(EPOCHS):
        model.train()
        for img, bio, yb in train_loader:
            img, bio, yb = img.to(device), bio.to(device), yb.to(device)
            img = train_augment(img)
            optimizer.zero_grad()
            loss = criterion(model(img, bio), yb)
            loss.backward()
            optimizer.step()
 
    model.eval()
    with torch.no_grad():
        for img, bio, yb in val_loader:
            img, bio = img.to(device), bio.to(device)
            probs = torch.softmax(model(img, bio), dim=1).cpu().numpy()
            all_preds.extend(np.argmax(probs, axis=1))
            all_labels.extend(yb.numpy())
            all_probs.extend(probs[:, 1])
    print(f"Fold {fold + 1}/5 terminé.")

Fold 1/5 terminé.
Fold 2/5 terminé.
Fold 3/5 terminé.
Fold 4/5 terminé.
Fold 5/5 terminé.


In [9]:

print("\nRapport de classification (Nondemented=0, Converted=1) :")
print(classification_report(all_labels, all_preds, target_names=["Nondemented", "Converted"]))
print("Matrice de confusion :")
print(confusion_matrix(all_labels, all_preds))
print(f"AUC-ROC : {roc_auc_score(all_labels, all_probs):.3f}")
 
print("\nÀ comparer avec la version CNN seul (image uniquement) et la régression "
      "logistique seule (biomarqueurs uniquement) pour voir si la fusion apporte "
      "un vrai gain, ou si l'une des deux sources domine.")


Rapport de classification (Nondemented=0, Converted=1) :
              precision    recall  f1-score   support

 Nondemented       0.82      0.85      0.84        72
   Converted       0.08      0.07      0.08        14

    accuracy                           0.72        86
   macro avg       0.45      0.46      0.46        86
weighted avg       0.70      0.72      0.71        86

Matrice de confusion :
[[61 11]
 [13  1]]
AUC-ROC : 0.422

À comparer avec la version CNN seul (image uniquement) et la régression logistique seule (biomarqueurs uniquement) pour voir si la fusion apporte un vrai gain, ou si l'une des deux sources domine.
